# Forward propagation

_Deep Learning_

**Push the input through every layer, left to right, to get the prediction.**

Forward propagation is how a network makes a prediction.

     You feed the input into the first layer. Its output becomes the input to the next layer. And so on, until the last layer.

---

This notebook is a practice scaffold for the **Forward propagation** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We'll build a tiny 2-layer network and push a single example through it. A forward pass is just function composition: each layer transforms its input and hands the result to the next. We do it in two steps — first define the network, then run the input through it.

### Step 1 — Define the network architecture

A `Sequential` model stacks layers in order. Here the input has 3 features; the first `Linear` layer maps those 3 numbers to 4 hidden units, a `ReLU` adds the nonlinearity that lets the network model non-straight relationships, and the final `Linear` collapses the 4 hidden units down to a single output. Defining the network does **not** run any data through it yet — it just sets up the weights.

In [ ]:
import torch
import torch.nn as nn

# 3 inputs -> 4 hidden (ReLU) -> 1 output
net = nn.Sequential(
    nn.Linear(3, 4),   # layer 1: weights W1, bias b1
    nn.ReLU(),         # activation g(.)
    nn.Linear(4, 1),   # layer 2: weights W2, bias b2
)

### Step 2 — Run one example forward

Now we feed a single example (3 features) into the network. Calling `net(x)` runs every layer left to right — Linear, ReLU, Linear — and returns the prediction. The output has shape `[1, 1]`: one example, one output value.

In [ ]:
x = torch.tensor([[2.0, -1.0, 0.5]])   # one example, 3 features
y = net(x)                              # run all layers left to right

print("output shape:", y.shape)         # torch.Size([1, 1])
print("prediction:", y.item())          # the network's answer

## Visualize the data & results

_After a forward pass, do real handwritten 0s and 1s separate into two clusters?_

### Step 1 — Load the digit images and keep just 0s and 1s

We reload the digits dataset, scale the pixel values from 0–16 down to 0–1, and keep only the examples labelled 0 or 1. Working with two classes keeps the picture readable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

digits = load_digits()
X = digits.data / 16.0
y = digits.target

mask = np.isin(y, [0, 1])               # keep real 0 and 1 images

### Step 2 — Project to 2-D with PCA

Each image has 64 pixel features — too many to plot directly. PCA compresses those 64 dimensions down to the 2 that capture the most variation, so we can place every image as a point on a flat scatter plot.

In [ ]:
X2 = PCA(n_components=2, random_state=0).fit_transform(X[mask])
y2 = y[mask]

### Step 3 — Plot the two clusters

We colour each point by its true digit. If the pixel patterns of 0s and 1s are genuinely different, the two colours should land in separate regions — which is exactly the kind of separation a forward pass through a trained network would exploit.

In [ ]:
plt.scatter(X2[y2 == 0][:, 0], X2[y2 == 0][:, 1], color="#4ea1ff", label="digit 0")
plt.scatter(X2[y2 == 1][:, 0], X2[y2 == 1][:, 1], color="#7ee787", label="digit 1")
plt.title("Real digit images (0 vs 1) after forward pass, PCA to 2-D")
plt.xlabel("component 1"); plt.ylabel("component 2")
plt.legend()
plt.show()